# p1 — Model Score Collection

**Purpose:** Walk the raw Open LLM Leaderboard result files in `data_raw/results/` and extract per-model benchmark scores into clean Parquet files.

**How to use:** Set `BENCH` in the *Config* cell below to the benchmark you want to process, then run all cells. Two files are written to `data_raw_clean/`:
- `{BENCH}_scores_full.parquet` — every evaluation record (one row per model × evaluation date)
- `{BENCH}_scores_unique.parquet` — deduplicated: one row per model, keeping the most recent evaluation

**Supported values for `BENCH`:** `bbh`, `math`, `gpqa`, `musr`

**Input:** `data_raw/results/*/*/*.json` (raw leaderboard JSON files)  
**Output:** `data_raw_clean/{BENCH}_scores_full.parquet`, `data_raw_clean/{BENCH}_scores_unique.parquet`

## Config

Set `BENCH` to select which benchmark to extract. All other cells adapt automatically.

In [1]:
BENCH = "bbh"   # one of: bbh | math | gpqa | musr

## Task Definitions

Each benchmark is a list of `(task_key, metric_key, short_name)` tuples that map to keys inside the leaderboard JSON files:
- `task_key` — the key under `results` in the JSON (e.g. `"leaderboard_bbh_snarks"`)
- `metric_key` — the metric field to extract (e.g. `"acc_norm,none"`)
- `short_name` — the column name written to the output Parquet

BBH uses `acc_norm,none`; MATH uses `exact_match,none`; GPQA and MuSR use `acc_norm,none`.

In [3]:
import glob
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

BBH_TASKS = [
    (f"leaderboard_bbh_{t}", "acc_norm,none", t)
    for t in [
        "boolean_expressions",
        "causal_judgement",
        "date_understanding",
        "disambiguation_qa",
        "formal_fallacies",
        "geometric_shapes",
        "hyperbaton",
        "logical_deduction_five_objects",
        "logical_deduction_seven_objects",
        "logical_deduction_three_objects",
        "movie_recommendation",
        "navigate",
        "object_counting",
        "penguins_in_a_table",
        "reasoning_about_colored_objects",
        "ruin_names",
        "salient_translation_error_detection",
        "snarks",
        "sports_understanding",
        "temporal_sequences",
        "tracking_shuffled_objects_five_objects",
        "tracking_shuffled_objects_seven_objects",
        "tracking_shuffled_objects_three_objects",
        "web_of_lies",
    ]
]

MATH_TASKS = [
    (f"leaderboard_math_{t}_hard", "exact_match,none", f"math_{t}")
    for t in [
        "algebra",
        "counting_and_prob",
        "geometry",
        "intermediate_algebra",
        "num_theory",
        "prealgebra",
        "precalculus",
    ]
]

GPQA_TASKS = [
    ("leaderboard_gpqa_diamond",  "acc_norm,none", "gpqa_diamond"),
    ("leaderboard_gpqa_extended", "acc_norm,none", "gpqa_extended"),
    ("leaderboard_gpqa_main",     "acc_norm,none", "gpqa_main"),
]

MUSR_TASKS = [
    ("leaderboard_musr_murder_mysteries", "acc_norm,none", "musr_murder_mysteries"),
    ("leaderboard_musr_object_placements", "acc_norm,none", "musr_object_placements"),
    ("leaderboard_musr_team_allocation",  "acc_norm,none", "musr_team_allocation"),
]

TASK_MAP = {
    "bbh":  BBH_TASKS,
    "math": MATH_TASKS,
    "gpqa": GPQA_TASKS,
    "musr": MUSR_TASKS,
}

tasks = TASK_MAP[BENCH]
print(f"Benchmark: {BENCH} — {len(tasks)} subtasks")

Benchmark: bbh — 24 subtasks


## Score Collection

For each JSON file under `data_raw/results/`, we:
1. Skip files not matching the `results_<date>T...` naming pattern (metadata / config files)
2. Parse `model_name` and evaluation date from the filename
3. Extract the score for each subtask from `data["results"][task_key][metric_key]`; missing values become `NaN`

Models that were evaluated multiple times will appear as multiple rows in `_full.parquet` — deduplication happens in the next step.

In [7]:
def extract_date(path: str) -> str:
    """Pull the evaluation date from filenames like 'results_2024-07-01T...'."""
    match = re.search(r"results_(\d{4}-\d{2}-\d{2})T", Path(path).name)
    return match.group(1) if match else None


rows = []
skipped = 0

for path in glob.glob("../data_raw/results/*/*/*.json"):
    print('collect from ', path)
    if "results_" not in Path(path).name:
        skipped += 1
        continue

    try:
        with open(path) as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError):
        skipped += 1
        continue

    result_scores = data.get("results", {})
    row = {
        "model_name":      data.get("model_name"),
        "evaluation_date": extract_date(path),
    }
    for task_key, metric_key, short in tasks:
        row[short] = result_scores.get(task_key, {}).get(metric_key)

    rows.append(row)

df = pd.DataFrame(rows)
print(f"Collected {len(df):,} records ({skipped} files skipped)")
print(df.dtypes)

collect from  ../data_raw/results/migtissera/Llama-3-70B-Synthia-v3.5/results_2024-10-24T00-00-00.000000.json
collect from  ../data_raw/results/migtissera/Llama-3-70B-Synthia-v3.5/results_2025-02-13T18-27-04.338360.json
collect from  ../data_raw/results/migtissera/Llama-3-70B-Synthia-v3.5/results_2024-07-18T03-07-17.091809.json
collect from  ../data_raw/results/migtissera/Trinity-2-Codestral-22B/results_2024-09-18T14-57-47.111766.json
collect from  ../data_raw/results/migtissera/Trinity-2-Codestral-22B/results_2024-10-24T00-00-00.000000.json
collect from  ../data_raw/results/migtissera/Trinity-2-Codestral-22B/results_2025-02-13T18-27-04.338360.json
collect from  ../data_raw/results/migtissera/Tess-3-7B-SFT/results_2024-10-24T00-00-00.000000.json
collect from  ../data_raw/results/migtissera/Tess-3-7B-SFT/results_2024-07-29T08-57-05.895142.json
collect from  ../data_raw/results/migtissera/Tess-3-7B-SFT/results_2025-02-13T18-27-04.338360.json
collect from  ../data_raw/results/migtissera/T

## Save Full Records

`_scores_full.parquet` retains every evaluation record, including duplicate runs of the same model. This is useful for auditing evaluation dates or analyzing score drift across re-evaluations.

In [9]:
out_full = f"../data_raw_clean/{BENCH}_scores_full.parquet"
df.to_parquet(out_full, index=False)
print(f"Saved {len(df):,} rows → {out_full}")

Saved 10,504 rows → ../data_raw_clean/bbh_scores_full.parquet


## Deduplicate & Save Unique Records

Some models were evaluated more than once (e.g. after a leaderboard update). For IRT fitting we want one row per model. We sort by evaluation date and keep the **most recent** result for each `model_name`.

In [10]:
df["evaluation_date_ts"] = pd.to_datetime(df["evaluation_date"], errors="coerce")

cleaned = (
    df.sort_values("evaluation_date_ts")
    .drop_duplicates(subset="model_name", keep="last")
    .drop(columns="evaluation_date_ts")
    .reset_index(drop=True)
)

out_unique = f"../data_raw_clean/{BENCH}_scores_unique.parquet"
cleaned.to_parquet(out_unique, index=False)
print(f"Unique models: {len(cleaned):,} (removed {len(df) - len(cleaned):,} duplicates)")
print(f"Saved → {out_unique}")
cleaned.head()

Unique models: 4,506 (removed 5,998 duplicates)
Saved → ../data_raw_clean/bbh_scores_unique.parquet


,model_name,evaluation_date,boolean_expressions,causal_judgement,date_understanding,disambiguation_qa,formal_fallacies,geometric_shapes,hyperbaton,logical_deduction_five_objects,...,reasoning_about_colored_objects,ruin_names,salient_translation_error_detection,snarks,sports_understanding,temporal_sequences,tracking_shuffled_objects_five_objects,tracking_shuffled_objects_seven_objects,tracking_shuffled_objects_three_objects,web_of_lies
0,mistralai/Mixtral-8x22B-Instruct-v0.1,2024-06-26,0.844,0.582888,0.696,0.768,0.620,0.520,0.812,0.624,...,0.672,0.848,0.588,0.702247,0.684,0.788,0.272,0.276,0.384,0.488
1,databricks/dbrx-instruct,2024-06-27,0.828,0.625668,0.516,0.712,0.612,0.264,0.652,0.420,...,0.556,0.788,0.548,0.606742,0.844,0.428,0.176,0.120,0.248,0.476
2,Qwen/Qwen2-57B-A14B,2024-07-04,0.900,0.609626,0.600,0.604,0.620,0.528,0.848,0.416,...,0.572,0.672,0.456,0.741573,0.812,0.736,0.140,0.124,0.300,0.488
3,google/gemma-2-2b-it,2024-07-29,0.740,0.566845,0.460,0.476,0.480,0.372,0.580,0.276,...,0.360,0.344,0.388,0.578652,0.556,0.116,0.180,0.140,0.344,0.488
4,rasyosef/phi-2-instruct-v0.1,2024-08-12,0.824,0.566845,0.472,0.604,0.516,0.104,0.732,0.488,...,0.388,0.508,0.460,0.713483,0.532,0.212,0.136,0.108,0.308,0.528
